# NB-FLUX spin-resolved preflight

This notebook replaces the old pilot; it is not a continuation of it. It verifies the
fixed-radius, spin-resolved measurement machinery before any A100 physics allocation.

It contains no continuum mass claim. A green result means the bounded certificates passed:
the J=1/J=3 sources, contaminant basis, fine-lattice p=0 projection, spatial flow,
checkpoint/restart, and shared correlated-analysis path all passed bounded tests.

Fixed physical flow radius controls source shape, not its multiplicative continuum
normalization. With this clover convention, the unrenormalized cubic-source weight carries an
expected kinematic factor proportional to `a^9 g^6`. The notebook uses separated-time Gram
whitening only as a rescaling-invariant diagnostic. It makes no absolute continuum-residue
claim without operator matching and source convergence.


In [1]:
from pathlib import Path
import json, platform, sys
import ipykernel, nbclient, nbformat

root = Path.cwd()
candidates = [
    root / "outputs" / "NB_FLUX_SPIN_RESOLVED_engine.py",
    root / "NB_FLUX_SPIN_RESOLVED_engine.py",
]
engine_path = next((path for path in candidates if path.exists()), None)
if engine_path is None:
    raise FileNotFoundError(
        "Place NB_FLUX_SPIN_RESOLVED_engine.py and NB_FLUX_T1PM_engine_repaired.py "
        "beside this notebook, or run from the WORKHOUSE outputs directory."
    )
sys.path.insert(0, str(engine_path.parent.resolve()))
import NB_FLUX_SPIN_RESOLVED_engine as engine

runtime = engine.runtime_manifest()
assert platform.python_version() == engine.APPROVED_PYTHON, runtime
assert runtime["numpy"] == engine.APPROVED_NUMPY, runtime
assert runtime["scipy"] == engine.APPROVED_SCIPY, runtime
assert nbformat.__version__ == engine.APPROVED_NBFORMAT
assert nbclient.__version__ == engine.APPROVED_NBCLIENT
assert ipykernel.__version__ == engine.APPROVED_IPYKERNEL
print("Engine:", engine.ENGINE_VERSION)
print("Engine hash:", engine.module_hash())
print("Base-engine hash:", engine.base_module_hash())
print(
    "Runtime:", runtime["python"], runtime["numpy"], runtime["scipy"],
    nbformat.__version__, nbclient.__version__, ipykernel.__version__
)


Engine: NB-FLUX spin-resolved preflight 2026-08-22
Engine hash: 676A2C19A5A817178903B6A7AC27A5D3FDC8C0B3080DC92EDB482456A43079D2
Base-engine hash: 258FEFD36B4155D1E96376A7FFCC4282B810BD2F1279B90DD58A8727FABCB42C
Runtime: 3.12.13 2.3.5 1.18.1 5.11.1 0.11.0 7.3.0


## Hard preflight

The finite ensemble used by the restart test is deliberately tiny. Its observables are discarded;
they exist only to prove interruption equivalence. The synthetic correlator test has a declared
mass and verifies that shared blocked resampling, GEVP reconstruction, full covariance, and a
correlated fit recover it.


In [2]:
report = engine.run_preflight(include_restart=True)
result_dir = root / "outputs" if (root / "outputs").exists() else root
result_path = result_dir / "NB_FLUX_SPIN_RESOLVED_preflight_result.json"
engine.strict_json_dump(result_path, report)
print("overall_passed =", report["overall_passed"])
print("saved =", result_path.resolve())



A. EXACT SPATIAL CARRIER CERTIFICATE
  [PASS] chain condition d2 d3 = 0: max exact entry=0
  [PASS] three harmonic plaquette planes: d2 H max=0, d3^T H max=0, b2=3
  [PASS] torus incidence ranks (exact modular certificate): rank(d2)=(126, 126), rank(d3)=(63, 63) over primes (1000003, 1000033)
  [PASS] translated cube boundaries have no k=0 carrier: max entry of d3*1=0
  [PASS] 24 proper cubic rotations: signed-permutation group O generated exactly
  [PASS] cube-boundary irrep: proper-rotation scalar and parity odd; with ImTr it is A1^{--}, not T1^{+-}
  [PASS] zero-momentum harmonic carrier representation: actual cochain permutation gives P_R H=H R for all 24 rotations and P H=H; ImTr is C-odd
  [PASS] measured-loop RPC projector: all P/R/S paths pass 24 rotations, P=+, and C=- exactly


overall_passed = True
saved = C:\Users\Alex\Documents\Codex\2026-08-21\https-github-com-ats314-workhouse-https\outputs\NB_FLUX_SPIN_RESOLVED_preflight_result.json


In [3]:
for row in report["gates"]:
    print(("PASS" if row["passed"] else "FAIL").ljust(5), row["name"])
    print("      ", json.dumps(row["detail"], sort_keys=True))
assert report["overall_passed"], "Preflight failed; do not benchmark or run a pilot."


PASS  repaired L=4 chain/cubic/loop regression
       {"passed": 8, "total": 8}
PASS  spin tensor decomposition and cubic covariance
       {"a2_norm_error": 5.329070518200751e-15, "charge_conjugation_error": 0.0, "decomposition_error": 3.552713678800501e-15, "t1_rotation_error": 5.329070518200751e-15, "t2_norm_error": 1.1368683772161603e-13}
PASS  straight-pair T1+A2+T2 ranks 3+0+3
       {"projector_completeness_error": 2.220446049250313e-16, "projector_cross_error": 2.2371143170757382e-17, "rank_A2": 0.0, "rank_T1": 3.0, "rank_T2": 3.0, "representation_homomorphism_error": 0.0, "subspace_invariance_error": 2.220446049250313e-16}
PASS  improved source cancels quintic term
       {"improved_finest_absolute_error": 2.5144636260386935e-19, "improved_remainder_power": 7.023281198719647, "raw_remainder_power": 4.999959184831198}
PASS  fine-lattice p=0 and momentum-shell phase cancellation
       {"momentum_star_closure_error": 0.0, "one_site_translation_zero_sum_error": 7.105427357601002e

## Implemented operator registry

At two retained radii the production registry contains raw and quintic-improved plaquette sources,
pure-tensor `V_T1`, `H_T1`, `H_A2`, and `H_T2`, an `A1++` scalar stream,
total-momentum-zero two-glueball products, straight center-neutral torelon pairs, and string-scale
torelons. It also stores a C-even planar leakage monitor and the exact p=0 `A1--` cube-boundary
null. The straight-pair `A2` rank is exactly zero by symmetry; `A2` contamination is covered by
the two-glueball products instead.


In [4]:
for channel, labels in report["operator_labels"].items():
    rows = {"T1": 3, "A2": 1, "T2": 3, "A1": 1, "torelon": 3, "controls": 3}[channel]
    print(f"{channel:7s}: {len(labels):2d} multiplets, {len(labels)*rows:2d} row-time streams")
    for label in labels:
        print("   ", label)


T1     : 13 multiplets, 39 row-time streams
    raw_P_rho0.000000_tau0.00000000
    improved_P_rho0.000000_tau0.00000000
    V_T1_rho0.000000_tau0.00000000
    H_T1_rho0.000000_tau0.00000000
    raw_P_rho0.100000_tau0.01041667
    improved_P_rho0.100000_tau0.01041667
    V_T1_rho0.100000_tau0.01041667
    H_T1_rho0.100000_tau0.01041667
    A1pp_x_V_T1_n20_rho0.100000_tau0.01041667
    A1pp_x_V_T1_n21_rho0.100000_tau0.01041667
    A1pp_x_H_T1_n20_rho0.100000_tau0.01041667
    A1pp_x_H_T1_n21_rho0.100000_tau0.01041667
    ditorelon_T1_r2_1_rho0.100000_tau0.01041667
A2     :  4 multiplets,  4 row-time streams
    H_A2_rho0.000000_tau0.00000000
    H_A2_rho0.100000_tau0.01041667
    A1pp_x_H_A2_n20_rho0.100000_tau0.01041667
    A1pp_x_H_A2_n21_rho0.100000_tau0.01041667
T2     :  5 multiplets, 15 row-time streams
    H_T2_rho0.000000_tau0.00000000
    H_T2_rho0.100000_tau0.01041667
    A1pp_x_H_T2_n20_rho0.100000_tau0.01041667
    A1pp_x_H_T2_n21_rho0.100000_tau0.01041667
    ditorelon_T2_r

## Fixed physical radii

Spatial Wilson flow is performed independently on each Euclidean time slice, with

`R_d/a = sqrt(6 tau_s)` and `tau_s = rho^2 / [6 (a sqrt(sigma))^2]`.

The tuning-only pilot scans ρ = 0.30, 0.45, 0.60, 0.80, then freezes two adjacent radii before
fresh chains. The table below previews the default ρ = 0.45, 0.60 workload.


In [5]:
published_scales = {
    5.8941: 0.26118,
    6.0625: 0.19472,
    6.2350: 0.15003,
    6.5000: 0.10383,
}
for beta, scale in published_scales.items():
    points = engine.fixed_radius_schedule((0.45, 0.60), scale)
    row = ", ".join(f"rho={p.requested_rho:.2f}: tau={p.flow_time:.3f}" for p in points)
    print(f"beta={beta:.4f}, a*sqrt(sigma)={scale:.5f} -> {row}")


beta=5.8941, a*sqrt(sigma)=0.26118 -> rho=0.45: tau=0.495, rho=0.60: tau=0.880
beta=6.0625, a*sqrt(sigma)=0.19472 -> rho=0.45: tau=0.890, rho=0.60: tau=1.582
beta=6.2350, a*sqrt(sigma)=0.15003 -> rho=0.45: tau=1.499, rho=0.60: tau=2.666
beta=6.5000, a*sqrt(sigma)=0.10383 -> rho=0.45: tau=3.131, rho=0.60: tau=5.566


## A100 decision gate

The first GPU action is a **performance microbenchmark**, not a physics chain. It measures update,
spatial-flow, topology, operator, I/O, and memory costs on one representative volume. Keep the flag
below `False` on CPU and in ordinary notebook review. On an A100 with an installed CUDA/CuPy
candidate backend, set it to `True`; the benchmark records the exact CuPy, CUDA runtime, driver,
and device metadata. Freeze CuPy and the CUDA runtime before the profile-scale gate; the driver and
device remain recorded provenance. The benchmark still produces no physics result.


In [6]:
RUN_A100_BENCHMARK = False
benchmark_path = result_dir / "NB_FLUX_A100_kernel_benchmark.json"
if RUN_A100_BENCHMARK:
    benchmark = engine.kernel_microbenchmark(
        L=16, Nt=20, update_cycles=2, flow_steps=4, repetitions=3, require_gpu=True
    )
    engine.strict_json_dump(benchmark_path, benchmark)
    print(json.dumps(benchmark, indent=2))
else:
    print("A100 benchmark intentionally not launched.")


A100 benchmark intentionally not launched.


## Actual-profile flow gate

After the timing result is accepted and its CuPy/CUDA versions are frozen, run this second gate at
the pilot's true maximum flow time on all three geometries. It repeats action monotonicity,
unitarity/determinant, gauge covariance, step halving, every source family, and four-dimensional
topology checks. A timing benchmark alone does not authorize a tuning chain.


In [7]:
RUN_PROFILE_FLOW_PREFLIGHT = False
LOCKED_CUPY_VERSION = None       # copy from benchmark["runtime"]["cupy"]
LOCKED_CUDA_RUNTIME = None       # copy from benchmark["runtime"]["cuda_runtime"]
if RUN_PROFILE_FLOW_PREFLIGHT:
    if LOCKED_CUPY_VERSION is None or LOCKED_CUDA_RUNTIME is None:
        raise RuntimeError("Freeze the benchmark CuPy/CUDA runtime before this gate.")
    profile_rows = []
    for L in (16, 20, 24):
        cfg = engine.SpinResolvedConfig(
            beta=6.0625, L=L, Nt=20, thermal_cycles=0, n_cfg=1,
            separation_cycles=1, prefer_gpu=True, campaign_id="a100-profile-flow",
            chain_id=0, chain_count=1,
            physical_radii_sqrt_sigma=(0.30, 0.45, 0.60, 0.80),
            contaminant_radius_sqrt_sigma=0.45, reference_asqrt_sigma=0.19472,
            reference_asqrt_sigma_error=0.00054,
            published_asqrt_sigma=0.19472, published_asqrt_sigma_error=0.00054,
            scale_source="Athenodorou-Teper-beta-6.0625-table",
            topology_radius_sqrt_sigma=0.50, loop_shapes=("P",), fit_tmax=4,
            enforce_runtime_lock=True, required_cupy_version=LOCKED_CUPY_VERSION,
            required_cuda_runtime=int(LOCKED_CUDA_RUNTIME),
        )
        row = engine.profile_flow_preflight(cfg, require_gpu=True)
        profile_rows.append(row)
        print("L=", L, "passed=", row["passed"], "tau_max=", row["maximum_spatial_flow_time"])
    if not all(row["passed"] for row in profile_rows):
        raise RuntimeError("Actual-profile flow gate failed; do not run the tuning pilot.")
    if not benchmark_path.exists():
        raise RuntimeError("The profile suite requires the saved A100 benchmark artifact.")
    profile_path = result_dir / "NB_FLUX_A100_profile_flow_preflight.json"
    engine.strict_json_dump(
        profile_path,
        {
            "schema": "nb-flux-profile-suite-v1",
            "overall_passed": all(row["passed"] for row in profile_rows),
            "locked_cupy_version": LOCKED_CUPY_VERSION,
            "locked_cuda_runtime": int(LOCKED_CUDA_RUNTIME),
            "benchmark_sha256": engine.sha256_file(benchmark_path),
            "rows": profile_rows,
        },
    )
else:
    print("Actual-profile A100 flow gate intentionally not launched.")


Actual-profile A100 flow gate intentionally not launched.


## Run decision

If every preflight gate is green, proceed only to the A100 microbenchmark. Price the proposed
β = 6.0625, Nt = 20, L = 16, 20, 24, four-chain tuning pilot from measured throughput.
Do not run the legacy pilot or six-volume continuum profiles. The local AMD 7900 XTX remains outside
this validated path because the delivered backend is CUDA/CuPy; ROCm support is a separate port and
regression project.
